# 🦜🔗 LangChain: Building LLM-Powered Applications from the Ground Up
**Deep Technical Blog — Data Science Internship | Innomatics Research Labs**

> 🔑 This notebook uses **Google Gemini API** (`gemini-1.5-flash`)

---

### 📋 Table of Contents
1. [Installation & Setup](#setup)
2. [Basic LLM Call](#llm)
3. [PromptTemplate Usage](#prompt)
4. [Simple LCEL Chain](#chain)
5. [Memory Example](#memory)
6. [Custom Tool Example](#custom-tool)
7. [Agent with Tool](#agent)
8. [RAG — Document Q&A](#rag)
9. [Bonus — RAG with PDF Upload](#pdf)

---
## 🔧 Step 1: Installation & Setup <a name='setup'></a>
Run this cell first. It installs all required packages.

In [ ]:
# Install all required packages
!pip install -q langchain langchain-google-genai langchain-community langchain-core
!pip install -q faiss-cpu pypdf tiktoken duckduckgo-search google-generativeai

In [ ]:
import os
from getpass import getpass

# 🔑 Enter your Google Gemini API key securely
# Get your key from: https://aistudio.google.com/app/apikey
os.environ['GOOGLE_API_KEY'] = getpass('Enter your Google Gemini API Key: ')

print('✅ Gemini API Key set successfully!')

In [ ]:
# Quick test — verify Gemini is working
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash', temperature=0.7)
response = llm.invoke('Say hello in one sentence.')
print('✅ Gemini connected:', response.content)

---
## 🤖 Section 2.1 — Basic LLM Call <a name='llm'></a>

**Concept:** LangChain wraps LLM providers behind a unified interface.  
A `ChatModel` takes a list of messages (system, human, AI) and returns a message response.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize the Gemini chat model
llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash', temperature=0.7)

# Create a structured message list
messages = [
    SystemMessage(content='You are a helpful Python tutor.'),
    HumanMessage(content='Explain list comprehensions in one sentence.')
]

# Invoke the model
response = llm.invoke(messages)
print('Response:', response.content)

---
## 📝 Section 2.2 — PromptTemplate Usage <a name='prompt'></a>

**Concept:** `ChatPromptTemplate` lets you define reusable prompt schemas with named variables.  
Prompts become first-class objects you can version, test, and share.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# Define a reusable template with named placeholders
template = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert in {domain}. Be concise.'),
    ('human', 'Explain {concept} with a real-world analogy.')
])

# Build a simple chain: prompt → model → parser
chain = template | ChatGoogleGenerativeAI(model='gemini-1.5-flash') | StrOutputParser()

# Invoke with domain and concept
result = chain.invoke({
    'domain': 'machine learning',
    'concept': 'gradient descent'
})
print('Result:', result)

In [ ]:
# Try a different domain and concept
result2 = chain.invoke({
    'domain': 'data engineering',
    'concept': 'data pipeline'
})
print('Result:', result2)

---
## 🔗 Section 2.3 — Simple LCEL Chain <a name='chain'></a>

**Concept:** LCEL (LangChain Expression Language) uses the `|` pipe operator to connect components.  
Each component receives the output of the previous one — building a multi-step pipeline.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# Two-step chain: translate first, then summarise
translate_prompt = ChatPromptTemplate.from_template(
    'Translate this to English: {text}'
)
summarise_prompt = ChatPromptTemplate.from_template(
    'Summarise in one sentence: {translated}'
)

llm    = ChatGoogleGenerativeAI(model='gemini-1.5-flash')
parser = StrOutputParser()

# Compose the full pipeline using LCEL pipe operator
chain = (
    {'translated': translate_prompt | llm | parser}
    | summarise_prompt
    | llm
    | parser
)

# Test with a Spanish sentence
result = chain.invoke({'text': 'La inteligencia artificial transforma la industria moderna.'})
print('Final Output:', result)

---
## 🧠 Section 2.4 — Memory Example <a name='memory'></a>

**Concept:** LLMs have no state between calls. Memory injects prior conversation turns into the prompt.  
`RunnableWithMessageHistory` is the modern LCEL-compatible memory approach.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# In-memory history store (use Redis or DynamoDB in production)
store = {}

def get_session_history(session_id: str):
    """Returns or creates a chat history for a given session."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Prompt injects history via MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a friendly tutor. Remember what the user tells you.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}')
])

chain = prompt | ChatGoogleGenerativeAI(model='gemini-1.5-flash')

# Wrap chain with message history
with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='question',
    history_messages_key='history'
)

cfg = {'configurable': {'session_id': 'user_42'}}

# Turn 1 — introduce name
response1 = with_history.invoke({'question': 'My name is Priya.'}, config=cfg)
print('Turn 1:', response1.content)

# Turn 2 — model should recall the name from memory
response2 = with_history.invoke({'question': 'What is my name?'}, config=cfg)
print('Turn 2:', response2.content)

# Turn 3 — continue the conversation
response3 = with_history.invoke({'question': 'What did I tell you earlier?'}, config=cfg)
print('Turn 3:', response3.content)

---
## 🛠️ Section 2.6 — Custom Tool Example <a name='custom-tool'></a>

**Concept:** The `@tool` decorator turns any Python function into an LLM-callable tool.  
The docstring is what the LLM reads to decide when to use it.

In [ ]:
from langchain_core.tools import tool

@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    '''Converts a temperature from Celsius to Fahrenheit.'''
    return f'{celsius * 9/5 + 32:.1f}°F'

@tool
def get_stock_price(ticker: str) -> str:
    '''Returns the current stock price for a given ticker symbol.'''
    # Mock data — replace with real API in production
    prices = {'AAPL': '182.50', 'GOOG': '141.30', 'MSFT': '378.90'}
    return f"${prices.get(ticker.upper(), 'Unknown ticker')}"

@tool
def word_count(text: str) -> str:
    '''Counts the number of words in a given text string.'''
    return f'{len(text.split())} words'

# Inspect tool metadata
print('Tool name    :', celsius_to_fahrenheit.name)
print('Description  :', celsius_to_fahrenheit.description)
print('Input schema :', celsius_to_fahrenheit.args)

# Direct tool calls
print('\n--- Direct Tool Calls ---')
print('37°C =', celsius_to_fahrenheit.invoke({'celsius': 37}))
print('AAPL =', get_stock_price.invoke({'ticker': 'AAPL'}))
print('Words:', word_count.invoke({'text': 'LangChain is a powerful GenAI framework'}))

---
## 🤖 Section 2.5 — Agent with Tool <a name='agent'></a>

**Concept:** An agent lets the LLM decide which tool to call at runtime.  
It uses a ReAct loop: `Thought → Action → Observation → Thought → ...` until it can answer.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Define tools
@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    '''Converts a temperature from Celsius to Fahrenheit.'''
    return f'{celsius * 9/5 + 32:.1f}°F'

@tool
def word_count(text: str) -> str:
    '''Counts the number of words in a given text string.'''
    return f'{len(text.split())} words'

@tool
def simple_calculator(expression: str) -> str:
    '''Evaluates a basic math expression like 25 * 4 + 10.'''
    try:
        result = eval(expression)
        return str(result)
    except:
        return 'Invalid expression'

tools = [celsius_to_fahrenheit, word_count, simple_calculator]
llm   = ChatGoogleGenerativeAI(model='gemini-1.5-flash', temperature=0)

# Prompt template for the agent
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Use the available tools when needed.'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}'),
])

# Create agent and executor
agent    = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Test 1: Temperature
print('--- Test 1: Temperature Conversion ---')
r1 = executor.invoke({'input': 'What is 100 degrees Celsius in Fahrenheit?'})
print('Answer:', r1['output'])

# Test 2: Math
print('\n--- Test 2: Calculator ---')
r2 = executor.invoke({'input': 'What is 125 * 8 + 200?'})
print('Answer:', r2['output'])

# Test 3: Word count
print('\n--- Test 3: Word Count ---')
r3 = executor.invoke({'input': 'How many words are in: "LangChain is an amazing open source framework for building LLM apps"?'})
print('Answer:', r3['output'])

---
## 📄 Section 2.8 — RAG: Retrieval-Augmented Generation <a name='rag'></a>

**Concept:** RAG lets LLMs answer questions about documents too large for the context window.  
Documents are chunked → embedded → stored in FAISS → retrieved at query time.

> ⚠️ Gemini Embeddings require the `models/embedding-001` model.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.schema import Document

# Sample document (replace with PyPDFLoader for real PDFs)
sample_text = """
LangChain is an open-source framework designed to simplify the creation of applications
using large language models. It provides a standard interface for chains, agents, and
memory components. The framework supports integration with OpenAI, HuggingFace, Google
Gemini, and Anthropic models.

Key features of LangChain include:
1. Prompt Templates for reusable prompt management
2. Chains for connecting multiple LLM calls
3. Agents for autonomous tool-using reasoning
4. Memory for maintaining conversation context
5. Vector Stores for document retrieval (RAG)

LangChain was created by Harrison Chase in October 2022. It quickly became one of the
most popular AI frameworks with over 80,000 GitHub stars. The framework is particularly
useful for building chatbots, document Q&A systems, and research agents.

LCEL (LangChain Expression Language) was introduced to provide a declarative way to
compose chains using the pipe operator. It supports streaming, async execution, and
parallel processing out of the box.

Google Gemini is a multimodal AI model developed by Google DeepMind. It comes in three
sizes: Gemini Ultra, Gemini Pro, and Gemini Nano. Gemini 1.5 Flash is optimized for
speed and efficiency, making it ideal for high-frequency tasks.
"""

# Step 1: Create document and split into chunks
docs     = [Document(page_content=sample_text)]
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks   = splitter.split_documents(docs)
print(f'✅ Created {len(chunks)} chunks from document')

# Step 2: Embed using Google Generative AI Embeddings
embeddings  = GoogleGenerativeAIEmbeddings(model='models/embedding-001')
vectorstore = FAISS.from_documents(chunks, embeddings)
print('✅ Vector store created with FAISS + Gemini Embeddings')

# Step 3: Build the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatGoogleGenerativeAI(model='gemini-1.5-flash'),
    retriever=vectorstore.as_retriever(search_kwargs={'k': 3}),
    return_source_documents=True
)
print('✅ RAG chain ready\n')

# Step 4: Ask questions
questions = [
    'Who created LangChain and when?',
    'What is LCEL?',
    'What are the key features of LangChain?',
    'What is Gemini 1.5 Flash optimized for?'
]

for q in questions:
    result = qa_chain.invoke({'query': q})
    print(f'Q: {q}')
    print(f'A: {result["result"]}')
    print('-' * 60)

---
## 📂 Bonus — RAG with a Real PDF File <a name='pdf'></a>
Upload any PDF to Colab and query it with Gemini.

In [ ]:
# Upload a PDF using the Colab file upload widget
from google.colab import files
uploaded = files.upload()  # Select your PDF file here

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# Get the uploaded filename
filename = list(uploaded.keys())[0]
print(f'Loaded: {filename}')

# Load, chunk, embed, and index
loader      = PyPDFLoader(filename)
docs        = loader.load()
splitter    = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks      = splitter.split_documents(docs)
print(f'✅ {len(chunks)} chunks from {len(docs)} pages')

embeddings  = GoogleGenerativeAIEmbeddings(model='models/embedding-001')
vectorstore = FAISS.from_documents(chunks, embeddings)

qa_chain = RetrievalQA.from_chain_type(
    llm=ChatGoogleGenerativeAI(model='gemini-1.5-flash'),
    retriever=vectorstore.as_retriever(search_kwargs={'k': 4})
)

# Ask any question about your PDF
query  = input('\nAsk a question about your PDF: ')
answer = qa_chain.invoke({'query': query})
print('\nAnswer:', answer['result'])

---
## ✅ Summary

| # | Section | Model Used | Status |
|---|---------|-----------|--------|
| 1 | Basic LLM Call | gemini-1.5-flash | ✅ |
| 2 | PromptTemplate | gemini-1.5-flash | ✅ |
| 3 | LCEL Chain | gemini-1.5-flash | ✅ |
| 4 | Memory | gemini-1.5-flash | ✅ |
| 5 | Custom Tools | — (pure Python) | ✅ |
| 6 | Agent with Tools | gemini-1.5-flash | ✅ |
| 7 | RAG with FAISS | gemini-1.5-flash + embedding-001 | ✅ |
| 8 | RAG with PDF Upload | gemini-1.5-flash + embedding-001 | ✅ |

---
**Blog:** [Add your Medium link here]  
**GitHub:** [Add your GitHub link here]  
**Author:** [Your Name] | Innomatics Research Labs — Data Science Internship Feb 2026